# Fine-tune All Seven-Emotion Models (GoEmotions + YouTube-Domain)

This notebook handles the complete training pipeline:

**Part A — Stage 1:** Fine-tune `distilbert-base-uncased` on balanced GoEmotions seven-class data.

**Part B — Stage 2:** Continue training on YouTube-domain DeepSeek-labeled data with class-weighted loss.

**Part C — Multi-model:** Domain-adapt other benchmark models (SamLowe RoBERTa, j-hartmann DistilRoBERTa, j-hartmann RoBERTa-large) on the same YouTube-domain split.

**Part D — Upload:** Push all adapted models to Hugging Face.

Labels: `anger`, `disgust`, `fear`, `joy`, `neutral`, `sadness`, `surprise`

Recommended runtime: **T4 GPU or better**.

## 1. Install dependencies

In [33]:
!pip install -q "transformers>=4.41,<5.0.0" "datasets>=2.20" "accelerate>=0.30" "evaluate>=0.4" "scikit-learn>=1.5" "pandas>=2.2" "pyarrow>=15.0" "torch>=2.2" huggingface_hub

## 2. Clone or update the project repository

In [34]:
import os
from pathlib import Path

repo_url = "https://github.com/chasezhang1999/youtube-emotion-analyzer.git"
repo_dir = Path("/content/youtube-emotion-analyzer")

if repo_dir.exists():
    %cd /content/youtube-emotion-analyzer
    !git pull
else:
    %cd /content
    !git clone {repo_url}
    %cd /content/youtube-emotion-analyzer

print("Working directory:", Path.cwd())

/content/youtube-emotion-analyzer
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 566 bytes | 566.00 KiB/s, done.
From https://github.com/chasezhang1999/youtube-emotion-analyzer
   d742e1b..4c6550b  main       -> origin/main
Updating d742e1b..4c6550b
Fast-forward
 scripts/train_youtube_domain_all_models.py | 4 +++-
 1 file changed, 3 insertions(+), 1 deletion(-)
Working directory: /content/youtube-emotion-analyzer


## 3. Imports and setup

In [35]:
import inspect
import json
import random
import time

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    pipeline,
)

import datasets.config as datasets_config
datasets_config.TORCHVISION_AVAILABLE = False

SEED = 5240
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

TARGET_LABELS = ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]
LABEL_TO_ID = {label: i for i, label in enumerate(TARGET_LABELS)}
ID_TO_LABEL = {i: label for label, i in LABEL_TO_ID.items()}

# GitHub raw URLs for YouTube-domain data (used in Part B)
GITHUB_RAW = "https://raw.githubusercontent.com/chasezhang1999/youtube-emotion-analyzer/main"
YT_TRAIN_URL = f"{GITHUB_RAW}/data/youtube_domain_7class_deepseek/train.csv"
YT_VAL_URL = f"{GITHUB_RAW}/data/youtube_domain_7class_deepseek/validation.csv"

print("Labels:", TARGET_LABELS)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Labels: ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']
CUDA: True
GPU: Tesla T4


## 4. Helper functions

In [36]:
def make_training_args(**kwargs):
    sig = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" in sig:
        kwargs["eval_strategy"] = kwargs.pop("evaluation_strategy", "epoch")
    else:
        kwargs["evaluation_strategy"] = kwargs.pop("evaluation_strategy", "epoch")
    return TrainingArguments(**kwargs)


def make_trainer(model, args, train_dataset, eval_dataset, tokenizer, compute_metrics_fn, class_weights=None, early_stopping_patience=2):
    kwargs = {
        "model": model,
        "args": args,
        "train_dataset": train_dataset,
        "eval_dataset": eval_dataset,
        "compute_metrics": compute_metrics_fn,
    }
    sig = inspect.signature(Trainer.__init__).parameters
    if "processing_class" in sig:
        kwargs["processing_class"] = tokenizer
    elif "tokenizer" in sig:
        kwargs["tokenizer"] = tokenizer

    if early_stopping_patience > 0:
        kwargs["callbacks"] = [EarlyStoppingCallback(early_stopping_patience=early_stopping_patience)]

    if class_weights is not None:
        class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)
        if torch.cuda.is_available():
            class_weights_tensor = class_weights_tensor.cuda()

        class WeightedTrainer(Trainer):
            def compute_loss(self, model, inputs, return_outputs=False, **kw):
                labels = inputs.pop("labels")
                outputs = model(**inputs)
                logits = outputs.logits
                loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)
                loss = loss_fn(logits, labels)
                return (loss, outputs) if return_outputs else loss

        return WeightedTrainer(**kwargs)

    return Trainer(**kwargs)


def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    wp, wr, wf1, _ = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)
    mp, mr, mf1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    return {"accuracy": acc, "weighted_f1": wf1, "macro_f1": mf1,
            "weighted_precision": wp, "weighted_recall": wr,
            "macro_precision": mp, "macro_recall": mr}


def compute_class_weights_from_df(df):
    counts = df["label"].value_counts().sort_index().values.astype(float)
    weights = counts.sum() / (len(TARGET_LABELS) * counts)
    weights = weights / weights.min()
    return weights.tolist()


print("Helper functions loaded.")

Helper functions loaded.


---
## Part A — Stage 1: GoEmotions Fine-tuning

Train DistilBERT from scratch on balanced GoEmotions seven-class data.

## 5. Load GoEmotions dataset

Keep single-label samples for our 7 target emotions, then balance by downsampling.

In [37]:
PARQUET_URLS = {
    "train": "https://huggingface.co/datasets/SetFit/go_emotions/resolve/refs%2Fconvert%2Fparquet/default/train/0000.parquet",
    "validation": "https://huggingface.co/datasets/SetFit/go_emotions/resolve/refs%2Fconvert%2Fparquet/default/validation/0000.parquet",
    "test": "https://huggingface.co/datasets/SetFit/go_emotions/resolve/refs%2Fconvert%2Fparquet/default/test/0000.parquet",
}

def filter_single_target(df):
    label_cols = [c for c in df.columns if c != "text"]
    single = df[df[label_cols].sum(axis=1) == 1].copy()
    target = single[single[TARGET_LABELS].sum(axis=1) == 1].copy()
    target["label_name"] = target[TARGET_LABELS].idxmax(axis=1)
    target["label"] = target["label_name"].map(LABEL_TO_ID).astype(int)
    return target[["text", "label_name", "label"]].reset_index(drop=True)

go_frames = {}
for split, url in PARQUET_URLS.items():
    raw = pd.read_parquet(url)
    filtered = filter_single_target(raw)
    go_frames[split] = filtered
    print(split, go_frames[split].shape)
    print(go_frames[split]["label_name"].value_counts().sort_index())
    print()

train (17166, 3)
label_name
anger        1025
disgust       498
fear          430
joy           853
neutral     12823
sadness       817
surprise      720
Name: count, dtype: int64

validation (2105, 3)
label_name
anger        109
disgust       61
fear          58
joy          106
neutral     1592
sadness       84
surprise      95
Name: count, dtype: int64

test (2160, 3)
label_name
anger        131
disgust       76
fear          65
joy           93
neutral     1606
sadness      102
surprise      87
Name: count, dtype: int64



## 6. Convert to HF Dataset and tokenize

In [38]:
go_dataset = DatasetDict({
    s: Dataset.from_pandas(df[["text", "label"]], preserve_index=False)
    for s, df in go_frames.items()
})

BASE_MODEL = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

go_tokenized = go_dataset.map(tokenize_batch, batched=True)
go_tokenized = go_tokenized.remove_columns(["text"])
go_tokenized

Map:   0%|          | 0/17166 [00:00<?, ? examples/s]

Map:   0%|          | 0/2105 [00:00<?, ? examples/s]

Map:   0%|          | 0/2160 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 17166
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 2105
    })
    test: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 2160
    })
})

## 7. Stage 1: Train DistilBERT on GoEmotions

In [39]:
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=len(TARGET_LABELS), id2label=ID_TO_LABEL, label2id=LABEL_TO_ID,
)

s1_args = make_training_args(
    output_dir="./s1-goemotions-results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=50,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
)

s1_trainer = make_trainer(model, s1_args, go_tokenized["train"], go_tokenized["validation"], tokenizer, compute_metrics, early_stopping_patience=0)

t0 = time.time()
s1_trainer.train()
print(f"\nStage 1 training time: {round(time.time() - t0, 2)}s")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Macro F1,Weighted Precision,Weighted Recall,Macro Precision,Macro Recall
1,0.450700,0.468132,0.850831,0.853833,0.707191,0.867889,0.850831,0.769435,0.686329
2,0.369500,0.424864,0.863658,0.864863,0.725282,0.867489,0.863658,0.724776,0.730303
3,0.263300,0.448500,0.863183,0.864735,0.722877,0.867528,0.863183,0.722773,0.726604



Stage 1 training time: 679.78s


## 8. Evaluate Stage 1

In [40]:
s1_val = s1_trainer.evaluate(go_tokenized["validation"])
s1_test = s1_trainer.evaluate(go_tokenized["test"])
print("Stage 1 validation:", {k: round(v, 4) for k, v in s1_val.items()})
print("Stage 1 test:      ", {k: round(v, 4) for k, v in s1_test.items()})

Stage 1 validation: {'eval_loss': 0.4249, 'eval_accuracy': 0.8637, 'eval_weighted_f1': 0.8649, 'eval_macro_f1': 0.7253, 'eval_weighted_precision': 0.8675, 'eval_weighted_recall': 0.8637, 'eval_macro_precision': 0.7248, 'eval_macro_recall': 0.7303, 'eval_runtime': 8.1252, 'eval_samples_per_second': 259.07, 'eval_steps_per_second': 16.246, 'epoch': 3.0}
Stage 1 test:       {'eval_loss': 0.4327, 'eval_accuracy': 0.863, 'eval_weighted_f1': 0.8621, 'eval_macro_f1': 0.7285, 'eval_weighted_precision': 0.862, 'eval_weighted_recall': 0.863, 'eval_macro_precision': 0.734, 'eval_macro_recall': 0.7258, 'eval_runtime': 8.74, 'eval_samples_per_second': 247.139, 'eval_steps_per_second': 15.446, 'epoch': 3.0}


## 9. Save Stage 1 model

In [41]:
S1_DIR = "./youtube-emotion-distilbert"
s1_trainer.save_model(S1_DIR)
tokenizer.save_pretrained(S1_DIR)
print("Saved to", S1_DIR)

samples = [
    "I love this video so much!",
    "This campaign makes me angry.",
    "I am shocked by this announcement.",
    "This is just a normal update.",
]
pipe1 = pipeline("text-classification", model=S1_DIR, tokenizer=S1_DIR)
pipe1(samples, truncation=True, return_token_type_ids=False)

Device set to use cuda:0


Saved to ./youtube-emotion-distilbert


[{'label': 'joy', 'score': 0.9553956389427185},
 {'label': 'anger', 'score': 0.927776575088501},
 {'label': 'surprise', 'score': 0.9608830213546753},
 {'label': 'neutral', 'score': 0.9940127730369568}]

---
## Part B — Stage 2: YouTube-Domain Adaptation (DistilBERT)

Continue training the Stage 1 model on YouTube-domain DeepSeek-labeled data with class-weighted loss.

## 10. Load YouTube-domain DeepSeek data

The training data was pre-built by `scripts/build_deepseek_training_data.py` from 8,000 DeepSeek AI-labeled YouTube comments.

In [42]:
yt_train = pd.read_csv(YT_TRAIN_URL)
yt_val = pd.read_csv(YT_VAL_URL)

# Ensure columns match expected format
for df in [yt_train, yt_val]:
    df["text"] = df["text"].astype(str)
    if "label_id" in df.columns:
        df["label"] = df["label_id"].astype(int)
    if "label_name" not in df.columns and "label" in df.columns:
        df["label_name"] = df["label"].map(ID_TO_LABEL) if df["label"].dtype == int else df["label"]

print(f"YouTube-domain train: {len(yt_train)}")
print(yt_train["label_name"].value_counts().sort_index())
print(f"\nYouTube-domain validation: {len(yt_val)}")
print(yt_val["label_name"].value_counts().sort_index())

YouTube-domain train: 3193
label_name
anger       571
disgust     234
fear        161
joy         571
neutral     571
sadness     571
surprise    514
Name: count, dtype: int64

YouTube-domain validation: 798
label_name
anger       143
disgust      58
fear         40
joy         143
neutral     143
sadness     143
surprise    128
Name: count, dtype: int64


## 11. Tokenize YouTube-domain data and compute class weights

In [43]:
yt_dataset = DatasetDict({
    "train": Dataset.from_pandas(yt_train[["text", "label"]], preserve_index=False),
    "validation": Dataset.from_pandas(yt_val[["text", "label"]], preserve_index=False),
})
yt_tokenized = yt_dataset.map(tokenize_batch, batched=True)
yt_tokenized = yt_tokenized.remove_columns(["text"])

class_weights = compute_class_weights_from_df(yt_train)
print("Class weights:")
for label, w in zip(TARGET_LABELS, class_weights):
    print(f"  {label}: {w:.2f}")

Map:   0%|          | 0/3193 [00:00<?, ? examples/s]

Map:   0%|          | 0/798 [00:00<?, ? examples/s]

Class weights:
  anger: 1.00
  disgust: 2.44
  fear: 3.55
  joy: 1.00
  neutral: 1.00
  sadness: 1.00
  surprise: 1.11


## 12. Stage 2: Domain adaptation training

In [44]:
domain_model = AutoModelForSequenceClassification.from_pretrained(
    S1_DIR, num_labels=len(TARGET_LABELS), id2label=ID_TO_LABEL, label2id=LABEL_TO_ID,
)

s2_args = make_training_args(
    output_dir="./s2-domain-results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=20,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
)

s2_trainer = make_trainer(
    domain_model, s2_args,
    yt_tokenized["train"], yt_tokenized["validation"],
    tokenizer, compute_metrics,
    class_weights=class_weights,
    early_stopping_patience=2,
)

t0 = time.time()
s2_trainer.train()
print(f"\nStage 2 training time: {round(time.time() - t0, 2)}s")

Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Macro F1,Weighted Precision,Weighted Recall,Macro Precision,Macro Recall
1,1.157800,1.180188,0.576441,0.578692,0.564670,0.597548,0.576441,0.560619,0.596921
2,0.955900,1.189388,0.596491,0.599223,0.587548,0.607389,0.596491,0.587609,0.594366
3,0.650000,1.266929,0.600251,0.600424,0.584020,0.607473,0.600251,0.597693,0.577586
4,0.449700,1.267057,0.602757,0.604766,0.589387,0.608455,0.602757,0.589439,0.591283
5,0.383800,1.298677,0.604010,0.604906,0.589007,0.608579,0.604010,0.590192,0.590589



Stage 2 training time: 300.5s


## 13. Evaluate and save Stage 2

In [45]:
s2_val = s2_trainer.evaluate(yt_tokenized["validation"])
print("Stage 2 YouTube-domain validation:", {k: round(v, 4) for k, v in s2_val.items()})

S2_DIR = "./youtube-emotion-distilbert-domain-adapted"
s2_trainer.save_model(S2_DIR)
tokenizer.save_pretrained(S2_DIR)
print("Saved to", S2_DIR)

Stage 2 YouTube-domain validation: {'eval_loss': 1.2671, 'eval_accuracy': 0.6028, 'eval_weighted_f1': 0.6048, 'eval_macro_f1': 0.5894, 'eval_weighted_precision': 0.6085, 'eval_weighted_recall': 0.6028, 'eval_macro_precision': 0.5894, 'eval_macro_recall': 0.5913, 'eval_runtime': 3.1112, 'eval_samples_per_second': 256.497, 'eval_steps_per_second': 16.071, 'epoch': 5.0}
Saved to ./youtube-emotion-distilbert-domain-adapted


## 14. Compare Stage 1 vs Stage 2

In [46]:
comparison = [
    "This launch is amazing and I want to buy it now!",
    "The brand response is terrible and people are angry.",
    "This safety ad is scary but important.",
    "I did not expect that ending at all.",
    "Just here to check the product details.",
    "This company should be ashamed of themselves.",
    "Watching from Ghana, love this!",
    "This made me cry so much.",
]

p1 = pipeline("text-classification", model=S1_DIR, tokenizer=S1_DIR)
p2 = pipeline("text-classification", model=S2_DIR, tokenizer=S2_DIR)

rows = []
for t in comparison:
    r1 = p1(t, truncation=True, return_token_type_ids=False)[0]
    r2 = p2(t, truncation=True, return_token_type_ids=False)[0]
    rows.append({"text": t, "stage1": r1["label"], "s1_conf": round(r1["score"], 3),
                 "domain": r2["label"], "dom_conf": round(r2["score"], 3)})
pd.DataFrame(rows)

Device set to use cuda:0
Device set to use cuda:0


,text,stage1,s1_conf,domain,dom_conf
0,This launch is amazing and I want to buy it now!,joy,0.729,joy,0.878
1,The brand response is terrible and people are ...,fear,0.676,anger,0.980
2,This safety ad is scary but important.,fear,0.942,fear,0.977
3,I did not expect that ending at all.,neutral,0.906,surprise,0.957
4,Just here to check the product details.,neutral,0.996,neutral,0.980
5,This company should be ashamed of themselves.,anger,0.568,anger,0.966
6,"Watching from Ghana, love this!",neutral,0.524,joy,0.984
7,This made me cry so much.,sadness,0.955,sadness,0.986


---
## Part C — Multi-model Domain Adaptation

Train additional benchmark models on the same YouTube-domain split for fair comparison.
Uses `scripts/train_youtube_domain_all_models.py` which handles class weights, early stopping, and per-class metrics.

Models:
- `SamLowe/roberta-base-go_emotions` (public GoEmotions RoBERTa)
- `j-hartmann/emotion-english-distilroberta-base` (public 7-emotion DistilRoBERTa)
- `j-hartmann/emotion-english-roberta-large` (larger RoBERTa, optional — needs more VRAM)

## 15. Log in to Hugging Face

Add your HF write token to Colab Secrets as `HF_TOKEN` before running.

In [47]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

## 16. Dry-run training configuration

In [48]:
!python scripts/train_youtube_domain_all_models.py --model all --hub-namespace chase1zhang --dry-run --train-csv data/youtube_domain_7class_deepseek/train.csv --validation-csv data/youtube_domain_7class_deepseek/validation.csv

{
  "selected_models": [
    {
      "key": "distilbert",
      "display_name": "GoEmotions fine-tuned DistilBERT",
      "base_model": "chase1zhang/youtube-emotion-distilbert",
      "output_slug": "youtube-emotion-distilbert-domain-adapted",
      "notes": "Project-owned DistilBERT baseline rerun on the shared YouTube-domain split."
    },
    {
      "key": "samlowe_roberta",
      "display_name": "Public GoEmotions RoBERTa (SamLowe)",
      "base_model": "SamLowe/roberta-base-go_emotions",
      "output_slug": "youtube-emotion-samlowe-roberta-domain-adapted",
      "notes": "Strong public GoEmotions RoBERTa adapted to the project seven-emotion taxonomy."
    },
    {
      "key": "jhartmann_distilroberta",
      "display_name": "Public DistilRoBERTa 7-emotion (j-hartmann)",
      "base_model": "j-hartmann/emotion-english-distilroberta-base",
      "output_slug": "youtube-emotion-jhartmann-distilroberta-domain-adapted",
      "notes": "Public seven-emotion baseline adapted to YouTub

## 17. Train smaller / medium models

These three models are the most important fair-comparison set. Each uses the same YouTube-domain split.

In [49]:
!python scripts/train_youtube_domain_all_models.py --model distilbert --hub-namespace chase1zhang --push-to-hub --epochs 3 --batch-size 16 --max-length 128 --train-csv data/youtube_domain_7class_deepseek/train.csv --validation-csv data/youtube_domain_7class_deepseek/validation.csv --fp16
!python scripts/train_youtube_domain_all_models.py --model samlowe_roberta --hub-namespace chase1zhang --push-to-hub --epochs 3 --batch-size 16 --max-length 128 --train-csv data/youtube_domain_7class_deepseek/train.csv --validation-csv data/youtube_domain_7class_deepseek/validation.csv --fp16
!python scripts/train_youtube_domain_all_models.py --model jhartmann_distilroberta --hub-namespace chase1zhang --push-to-hub --epochs 3 --batch-size 16 --max-length 128 --train-csv data/youtube_domain_7class_deepseek/train.csv --validation-csv data/youtube_domain_7class_deepseek/validation.csv --fp16

Training GoEmotions fine-tuned DistilBERT: chase1zhang/youtube-emotion-distilbert
Map: 100% 3193/3193 [00:00<00:00, 3554.38 examples/s]
Map: 100% 798/798 [00:00<00:00, 3642.10 examples/s]
{'loss': 1.6069, 'grad_norm': 13.206198692321777, 'learning_rate': 1.9200000000000003e-05, 'epoch': 0.12}
{'loss': 1.4171, 'grad_norm': 11.635740280151367, 'learning_rate': 1.8366666666666668e-05, 'epoch': 0.25}
{'loss': 1.3719, 'grad_norm': 10.061747550964355, 'learning_rate': 1.7533333333333337e-05, 'epoch': 0.38}
{'loss': 1.3189, 'grad_norm': 11.755772590637207, 'learning_rate': 1.67e-05, 'epoch': 0.5}
{'loss': 1.2788, 'grad_norm': 10.848334312438965, 'learning_rate': 1.586666666666667e-05, 'epoch': 0.62}
{'loss': 1.1886, 'grad_norm': 15.010526657104492, 'learning_rate': 1.5033333333333336e-05, 'epoch': 0.75}
{'loss': 1.1882, 'grad_norm': 14.510855674743652, 'learning_rate': 1.4200000000000001e-05, 'epoch': 0.88}
{'loss': 1.1642, 'grad_norm': 9.69455623626709, 'learning_rate': 1.3366666666666669e-0

## 18. Optional: train RoBERTa-large

Run only if Colab runtime and memory are sufficient. Uses smaller batch + gradient accumulation for T4.

In [50]:
!python scripts/train_youtube_domain_all_models.py --model jhartmann_roberta_large --hub-namespace chase1zhang --push-to-hub --epochs 3 --batch-size 4 --gradient-accumulation-steps 2 --max-length 128 --train-csv data/youtube_domain_7class_deepseek/train.csv --validation-csv data/youtube_domain_7class_deepseek/validation.csv --fp16

Training Public RoBERTa-large 7-emotion (j-hartmann): j-hartmann/emotion-english-roberta-large
tokenizer_config.json: 100% 328/328 [00:00<00:00, 2.17MB/s]
vocab.json: 798kB [00:00, 10.5MB/s]
merges.txt: 456kB [00:00, 5.00MB/s]
tokenizer.json: 1.36MB [00:00, 8.63MB/s]
special_tokens_map.json: 100% 239/239 [00:00<00:00, 1.86MB/s]
Map: 100% 3193/3193 [00:00<00:00, 6691.87 examples/s]
Map: 100% 798/798 [00:00<00:00, 6776.58 examples/s]
config.json: 1.03kB [00:00, 4.81MB/s]
pytorch_model.bin: 100% 1.42G/1.42G [00:53<00:00, 26.7MB/s]
model.safetensors:   0% 0.00/1.42G [00:00<?, ?B/s]
model.safetensors:   5% 67.1M/1.42G [00:01<00:25, 52.2MB/s]   
  0% 1/1200 [00:01<38:22,  1.92s/it]
model.safetensors:  19% 268M/1.42G [00:04<00:15, 72.9MB/s] 
  0% 3/1200 [00:03<21:17,  1.07s/it]
  0% 4/1200 [00:04<18:50,  1.06it/s]
model.safetensors:  33% 470M/1.42G [00:05<00:10, 90.8MB/s]
  0% 6/1200 [00:05<15:07,  1.32it/s]
  1% 7/1200 [00:06<13:42,  1.45it/s]
  1% 8/1200 [00:06<11:01,  1.80it/s]
  1% 9/1200

## 19. Inspect all training metrics

In [51]:
metrics_paths = sorted(Path("fine_tuned_model_files/youtube_domain_all_models").glob("*/youtube_domain_training_metrics.json"))
print(f"Found {len(metrics_paths)} metric files:\n")
for path in metrics_paths:
    data = json.loads(path.read_text())
    metrics = data.get("metrics", {})
    print(f"{path.parent.name}")
    print(f"  repo:       {data.get('repo_id')}")
    print(f"  accuracy:   {round(metrics.get('eval_accuracy', 0), 4)}")
    print(f"  macro_f1:   {round(metrics.get('eval_macro_f1', 0), 4)}")
    print(f"  elapsed:    {data.get('elapsed_seconds')}s")
    print()

Found 4 metric files:

youtube-emotion-distilbert-domain-adapted
  repo:       chase1zhang/youtube-emotion-distilbert-domain-adapted
  accuracy:   0.5927
  macro_f1:   0.587
  elapsed:    134.0627s

youtube-emotion-jhartmann-distilroberta-domain-adapted
  repo:       chase1zhang/youtube-emotion-jhartmann-distilroberta-domain-adapted
  accuracy:   0.6291
  macro_f1:   0.6126
  elapsed:    251.3771s

youtube-emotion-roberta-large-domain-adapted
  repo:       chase1zhang/youtube-emotion-roberta-large-domain-adapted
  accuracy:   0.6717
  macro_f1:   0.6571
  elapsed:    1035.0202s

youtube-emotion-samlowe-roberta-domain-adapted
  repo:       chase1zhang/youtube-emotion-samlowe-roberta-domain-adapted
  accuracy:   0.6516
  macro_f1:   0.6436
  elapsed:    255.2849s



---
## Part D — Upload DistilBERT models to Hugging Face

The multi-model training script (Part C) already pushes its own models. This section uploads the DistilBERT Stage 1 and Stage 2 models from Parts A/B.

In [52]:
repo1 = "chase1zhang/youtube-emotion-distilbert"
repo2 = "chase1zhang/youtube-emotion-distilbert-domain-adapted"

s1_trainer.model.push_to_hub(repo1)
tokenizer.push_to_hub(repo1)
print(f"Uploaded Stage 1: https://huggingface.co/{repo1}")

s2_trainer.model.push_to_hub(repo2)
tokenizer.push_to_hub(repo2)
print(f"Uploaded domain-adapted: https://huggingface.co/{repo2}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...tilbert/model.safetensors:   0%|          |  575kB /  268MB            

Uploaded Stage 1: https://huggingface.co/chase1zhang/youtube-emotion-distilbert


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapted/model.safetensors:   2%|1         | 4.02MB /  268MB            

Uploaded domain-adapted: https://huggingface.co/chase1zhang/youtube-emotion-distilbert-domain-adapted


## 20. Verify uploaded models

In [53]:
v1 = pipeline("text-classification", model=repo1, tokenizer=repo1)
v2 = pipeline("text-classification", model=repo2, tokenizer=repo2)
print("Stage 1:", v1(samples, truncation=True, return_token_type_ids=False))
print("Domain: ", v2(samples, truncation=True, return_token_type_ids=False))

config.json:   0%|          | 0.00/853 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Device set to use cuda:0


Stage 1: [{'label': 'joy', 'score': 0.9553956389427185}, {'label': 'anger', 'score': 0.927776575088501}, {'label': 'surprise', 'score': 0.9608830213546753}, {'label': 'neutral', 'score': 0.9940127730369568}]
Domain:  [{'label': 'joy', 'score': 0.9842528700828552}, {'label': 'anger', 'score': 0.9768864512443542}, {'label': 'surprise', 'score': 0.9736225008964539}, {'label': 'neutral', 'score': 0.9301170706748962}]


## 21. Summary

Copy these into the report:
- Stage 1 GoEmotions validation / test metrics (cell 8)
- Stage 2 YouTube-domain validation metrics (cell 13)
- Multi-model domain adaptation metrics (cell 19)
- Hugging Face model URLs

The Streamlit app default model: `chase1zhang/youtube-emotion-distilbert-domain-adapted`